##Build Results Fact
1. Read silver results table
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns
     - is_win -> Indicates that the driver own the race
     - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
     - has_points -> Indicates that the driver has scored points
6. Write the transformed data to gold fact_session_results table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
#Reading the source tables
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .filter(F.col("batch_id") == v_batch_id)
         .withColumn("session_type", F.lit("RACE"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .filter(F.col("batch_id") == v_batch_id)
         .withColumn("session_type", F.lit("SPRINT"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

In [0]:
#Union on the 2 tables (results_df and sprints_df)
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
# #Derive the columns is_win, is_podium, has_points
# is_win -> Indicates that the driver own the race
# is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
# has_points -> Indicates that the driver has scored points

fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season = 2025"))

season,round,constructor_id,driver_id,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
2025,1,mercedes,russell,4,57,63,15.0,3,3,Finished,RACE,false,true,true
2025,1,mercedes,antonelli,16,57,12,12.0,4,4,Finished,RACE,false,false,true
2025,1,ferrari,hamilton,8,57,44,1.0,10,10,Finished,RACE,false,false,true
2025,1,haas,ocon,19,57,31,0.0,13,13,Finished,RACE,false,false,false
2025,1,mclaren,piastri,2,57,81,2.0,9,9,Finished,RACE,false,false,true
2025,1,alpine,gasly,9,57,10,0.0,11,11,Finished,RACE,false,false,false
2025,1,haas,bearman,20,57,87,0.0,14,14,Finished,RACE,false,false,false
2025,1,red_bull,lawson,18,46,30,0.0,15,R,Retired,RACE,false,false,false
2025,1,aston_martin,alonso,12,32,14,0.0,17,R,Retired,RACE,false,false,false
2025,1,williams,albon,6,57,23,10.0,5,5,Finished,RACE,false,false,true


In [0]:
write_to_gold(
    input_df = fact_session_results_df,
    target_table = target_table,
    merge_condition="""
        t.season = s.season 
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
    """,
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "driver_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]
)

In [0]:
display(spark.table(target_table))

season,round,constructor_id,driver_id,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points,created_timestamp,updated_timestamp
2024,1,mercedes,russell,3,57,63,10.0,5,5,Finished,SPRINT,false,false,true,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,1,sauber,zhou,17,56,24,0.0,11,11,Lapped,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,2,rb,ricciardo,14,49,3,0.0,16,16,Lapped,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,2,aston_martin,stroll,10,5,18,0.0,19,R,Retired,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,4,rb,tsunoda,10,52,22,1.0,10,10,Lapped,SPRINT,false,false,true,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,6,mclaren,piastri,6,57,81,0.0,13,13,Finished,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,6,haas,kevin_magnussen,18,57,20,0.0,19,19,Finished,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,8,ferrari,sainz,3,78,55,15.0,3,3,Finished,SPRINT,false,true,true,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,8,rb,ricciardo,12,76,3,0.0,12,12,Lapped,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
2024,9,rb,tsunoda,8,70,22,0.0,14,14,Finished,SPRINT,false,false,false,2026-08-05T16:07:47.439Z,2026-08-05T16:09:11.912Z
